In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
import xarray as xr

dataset = "gpcp"

save_storage = Path("/media/dmbrmv/ssd_2b/GPCP/ParsedMonthly") / dataset
save_storage.mkdir(parents=True, exist_ok=True)
years = range(2007, 2025)
data_dir = Path("/media/dmbrmv/ssd_2b/GPCP")


In [6]:
with xr.open_dataset("/media/dmbrmv/ssd_2b/GPCP/ParsedMonthly/gpcp/2023_06.nc") as f:
    pass

f

<xarray.Dataset> Size: 764kB
Dimensions:  (time: 30, lat: 40, lon: 159)
Coordinates:
  * lat      (lat) float32 160B 42.0 43.0 44.0 45.0 46.0 ... 78.0 79.0 80.0 81.0
  * lon      (lon) float32 636B 20.0 21.0 22.0 23.0 ... 175.0 176.0 177.0 178.0
  * time     (time) datetime64[ns] 240B 2023-06-01 2023-06-02 ... 2023-06-30
Data variables:
    precip   (time, lat, lon) float32 763kB ...
Attributes: (12/45)
    Conventions:                CF-1.6, ACDD 1.3
    title:                      Global Precipitation Climatatology Project (G...
    source:                     /data1/GPCP_CDR/GPCP_Output/1DD//bin/202306/s...
    references:                 Huffman et al. 1997, http://dx.doi.org/10.117...
    history:                    1) 2023-09-07T14:51:23Z, Dr. Jian-Jian Wang, ...
    Metadata_Conventions:       CF-1.6, Unidata Dataset Discovery v1.0, NOAA ...
    ...                         ...
    metadata_link:              gov.noaa.ncdc:XXXXX
    product_version:            v01r03
    platform:                   GOES (Geostationary Operational Environmental...
    sensor:                     Imager, AIRS > Atmospheric Infrared Sounder, ...
    spatial_resolution:         1 degree
    comment:                    Processing computer: eagle2.umd.edu

In [5]:
era5_land = pd.read_csv(
    "/home/dmbrmv/Development/camels_ru/data/MeteoData/CamelsRU/era5_land/1001.csv",
    parse_dates=["date"],
    index_col="date",
)
mswep = pd.read_csv(
    "/home/dmbrmv/Development/camels_ru/data/MeteoData/CamelsRU/mswep/1001.csv",
    parse_dates=["date"],
    index_col="date",
)
gpcp = pd.read_csv(
    "/home/dmbrmv/Development/camels_ru/data/MeteoData/CamelsRU/gpcp/1001.csv",
    parse_dates=["date"],
    index_col="date",
)
gpcp["precip"].groupby(pd.Grouper(freq="YE")).sum()

date
2008-12-31    461.934233
2009-12-31    400.908647
2010-12-31    382.207656
2011-12-31    426.221122
2012-12-31    492.181240
2013-12-31    534.132950
2014-12-31    453.159473
2015-12-31    345.227348
2016-12-31    426.294336
2017-12-31    417.106739
2018-12-31    438.744030
2019-12-31    485.807544
2020-12-31    274.265875
2021-12-31    300.505829
2022-12-31    139.297551
2023-12-31    164.832842
Freq: YE-DEC, Name: precip, dtype: float64

In [4]:
for year in tqdm(years, desc="Processing years"):
    for month in tqdm(range(1, 13), desc="Processing months", leave=False):
        save_path = save_storage / f"{year:04d}_{month:02d}.nc"
        if save_path.exists():
            continue
        gpcp_files = list(data_dir.glob(f"*_{year}{month:02d}*.nc4"))
        if len(gpcp_files) == 0:
            print(f"No V3 files found for {year}-{month:02d}")
            gpcp_files = list(data_dir.glob(f"*_d{year}{month:02d}*.nc"))
            if len(gpcp_files) == 0:
                print(f"\nNo files found for {year}-{month:02d}, skipping...")
                continue
        with xr.open_mfdataset(gpcp_files, combine="by_coords", chunks={"time": 30}) as ds:
            try:
                ds = ds.sortby("time").squeeze().drop_dims("bnds")
            except ValueError:
                ds = ds.sortby("time").squeeze().drop_dims("nv")
                ds = ds.rename({"latitude": "lat", "longitude": "lon"})
            ds = (
                ds.where(ds.lat > 41.0, drop=True)
                .where(ds.lat < 82.0, drop=True)
                .where(ds.lon > 19.0, drop=True)
                .where(ds.lon < 179.0, drop=True)
            )
            ds.to_netcdf(save_path)

Processing years:   0%|          | 0/18 [00:00<?, ?it/s]

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

No V3 files found for 2007-01

No files found for 2007-01, skipping...
No V3 files found for 2007-02

No files found for 2007-02, skipping...
No V3 files found for 2007-03

No files found for 2007-03, skipping...
No V3 files found for 2007-04

No files found for 2007-04, skipping...
No V3 files found for 2007-05

No files found for 2007-05, skipping...
No V3 files found for 2007-06

No files found for 2007-06, skipping...
No V3 files found for 2007-07

No files found for 2007-07, skipping...
No V3 files found for 2007-08

No files found for 2007-08, skipping...
No V3 files found for 2007-09

No files found for 2007-09, skipping...
No V3 files found for 2007-10

No files found for 2007-10, skipping...
No V3 files found for 2007-11

No files found for 2007-11, skipping...
No V3 files found for 2007-12

No files found for 2007-12, skipping...


Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

...

/tmp/ipykernel_2827052/3286088456.py:13: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  with xr.open_mfdataset(gpcp_files, combine="by_coords", chunks={"time": 30}) as ds:


Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

No V3 files found for 2024-01

No files found for 2024-01, skipping...
No V3 files found for 2024-02

No files found for 2024-02, skipping...
No V3 files found for 2024-03

No files found for 2024-03, skipping...
No V3 files found for 2024-04

No files found for 2024-04, skipping...
No V3 files found for 2024-05

No files found for 2024-05, skipping...
No V3 files found for 2024-06

No files found for 2024-06, skipping...
No V3 files found for 2024-07

No files found for 2024-07, skipping...
No V3 files found for 2024-08

No files found for 2024-08, skipping...
No V3 files found for 2024-09

No files found for 2024-09, skipping...
No V3 files found for 2024-10

No files found for 2024-10, skipping...
No V3 files found for 2024-11

No files found for 2024-11, skipping...
No V3 files found for 2024-12

No files found for 2024-12, skipping...


In [ ]:
with xr.open_dataset(
    "/media/dmbrmv/ssd_2b/GPCP/gpcp_v01r03_daily_d20080101_c20170530.area-subset.82.179.40.19.nc"
) as ds:
    ds = ds.rename({"latitude": "lat", "longitude": "lon"})
    ds = (
        ds.where(ds.lat > 41.0, drop=True)
        .where(ds.lat < 82.0, drop=True)
        .where(ds.lon > 19.0, drop=True)
        .where(ds.lon < 179.0, drop=True)
    )
ds

<xarray.Dataset> Size: 230kB
Dimensions:      (lat: 40, nv: 2, lon: 159, time: 1)
Coordinates:
  * lat          (lat) float32 160B 42.0 43.0 44.0 45.0 ... 78.0 79.0 80.0 81.0
  * lon          (lon) float32 636B 20.0 21.0 22.0 23.0 ... 176.0 177.0 178.0
  * time         (time) datetime64[ns] 8B 2008-01-01
Dimensions without coordinates: nv
Data variables:
    lat_bounds   (lat, nv, lon) float32 51kB 42.0 42.0 42.0 ... 82.0 82.0 82.0
    lon_bounds   (lon, nv, lat) float32 51kB 20.0 20.0 20.0 ... 179.0 179.0
    time_bounds  (time, nv, lat, lon) datetime64[ns] 102kB 2008-01-01 ... 200...
    precip       (time, lat, lon) float32 25kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes: (12/45)
    Conventions:                CF-1.6, ACDD 1.3
    title:                      Global Precipitation Climatatology Project (G...
    source:                     /data1/GPCP_CDR/GPCP_Output/1DD//bin/200801/s...
    references:                 Huffman et al. 1997, http://dx.doi.org/10.117...
    history:                    1) 2017-05-30T17:07:07Z, Dr. Jian-Jian Wang, ...
    Metadata_Conventions:       CF-1.6, Unidata Dataset Discovery v1.0, NOAA ...
    ...                         ...
    metadata_link:              gov.noaa.ncdc:XXXXX
    product_version:            v01r03
    platform:                   GOES (Geostationary Operational Environmental...
    sensor:                     Imager, AIRS > Atmospheric Infrared Sounder, ...
    spatial_resolution:         1 degree
    comment:                    Processing computer: eagle2.umd.edu

In [19]:
with xr.open_dataset("/media/dmbrmv/ssd_2b/GPCP/ParsedMonthly/gpcp/2008_01.nc") as f:
    pass

f["precip"]

<xarray.DataArray 'precip' (time: 31, lat: 82, lon: 320)> Size: 3MB
[813440 values with dtype=float32]
Coordinates:
  * lon      (lon) float32 1kB 19.25 19.75 20.25 20.75 ... 177.8 178.2 178.8
  * lat      (lat) float32 328B 81.75 81.25 80.75 80.25 ... 42.25 41.75 41.25
  * time     (time) datetime64[ns] 248B 2008-01-01 2008-01-02 ... 2008-01-31
Attributes:
    units:         mm/day
    long_name:     precipitation
    valid_range:   [  0. 500.]
    cell_methods:  time: mean